# 🌊 3D Visualization of Observed Floods

This notebook visualizes observed flood events in 3D using.  
We use orthophoto basemap and extruded polygons to show how many times each area was detected flooded.

## Install and import libraries

In [1]:
import importlib, sys, subprocess

# Mapovanie modul → pip balík
packages = {
    "shapely": "shapely",
    "pystac_client": "pystac-client",
    "odc": "odc-stac",
    "pyproj": "pyproj",
    "rioxarray": "rioxarray",
    "xarray": "xarray",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "rasterio": "rasterio",
    "ipywidgets": "ipywidgets",
    "ipyleaflet": "ipyleaflet",
    "leafmap": "leafmap",
}

to_install = []
for module, pip_name in packages.items():
    if importlib.util.find_spec(module) is None:
        to_install.append(pip_name)

if to_install:
    print("⏳ Installing missing packages... please wait.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)
else:
    print("✅ All required packages already installed.")

# --- Test importov ---
print("🔎 Testing imports...")
try:
    from shapely.geometry import box, shape, mapping
    from pystac_client import Client
    import odc.stac
    import pyproj
    import rioxarray
    import xarray as xr
    import numpy as np
    import geopandas as gpd
    import rasterio
    from rasterio.features import shapes
    import os, json, math
    import ipywidgets as widgets
    from ipyleaflet import Map, DrawControl, Rectangle
    from datetime import date, timedelta
    import leafmap.maplibregl as leafmap
    print("✅ All imports successful!")
except Exception as e:
    print("❌ Import error:", e)


✅ All required packages already installed.
🔎 Testing imports...
✅ All imports successful!


In [1]:
# === IMPORTS ===
from shapely.geometry import box, shape, mapping
from pystac_client import Client
import odc.stac
import pyproj
import rioxarray
import xarray as xr
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import shapes
import os
import json, math
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, Rectangle
from datetime import date, timedelta

import leafmap.maplibregl as leafmap

## Draw an AOI (rectangle) on the map

- Only rectangles are allowed (no polygons/lines).
- A small **size cap** is enforced (sum of width+height ≤ `MAX_SIZE` degrees) to prevent huge AOIs.
- The selected bbox is printed below the map.


In [2]:
# Create map
m = Map(center=(48.5, 11.0), zoom=9)

# Accept only rectangle
draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#3388ff", "fillOpacity": 0.1}},
    polygon={}, circle={}, circlemarker={}, polyline={}
)

output = widgets.Output()

# limit sum (width + height)
MAX_SIZE = 2

# ---- Default bbox ----
default_bbox = (10.5,48.5,10.9,48.7746)  # (minx, miny, maxx, maxy)

# save to variabel
selected_bbox = {
    "minx": default_bbox[0],
    "miny": default_bbox[1],
    "maxx": default_bbox[2],
    "maxy": default_bbox[3]
}

# draw to map like rectangle
last_rect = Rectangle(
    bounds=((default_bbox[1], default_bbox[0]), (default_bbox[3], default_bbox[2])),
    color="green",
    fill_opacity=0.2
)
m.add_layer(last_rect)

with output:
    print("Default bounding box:", selected_bbox)

# ---- Callback for new bbox ----
def handle_draw(_, action, geo_json):
    global selected_bbox, last_rect

    # delete old box
    if last_rect in m.layers:
        m.remove_layer(last_rect)

    if geo_json["geometry"]["type"] == "Polygon":
        geom = shape(geo_json["geometry"])
        minx, miny, maxx, maxy = geom.bounds
        width = maxx - minx
        height = maxy - miny
        size = width + height

        with output:
            output.clear_output()
            if size > MAX_SIZE:
                print(f"⚠️ Bounding box is too big! "
                      f"Max (width+height): {MAX_SIZE}°, "
                      f"currently: {size:.3f}° (width {width:.3f}°, height {height:.3f}°)")
                selected_bbox = {}
            else:
                selected_bbox = {
                    "minx": minx,
                    "miny": miny,
                    "maxx": maxx,
                    "maxy": maxy
                }
                # draw new rectangle
                last_rect = Rectangle(
                    bounds=((miny, minx), (maxy, maxx)),
                    color="red",
                    fill_opacity=0.2
                )
                m.add_layer(last_rect)

                print("Selected bounding box:", selected_bbox)

draw_control.on_draw(handle_draw)
m.add_control(draw_control)

display(m, output)


Map(center=[48.5, 11.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

Output()


## Pick a date range


In [3]:
# DatePicker start and end
start_picker = widgets.DatePicker(
    description="From:",
    value=date(2024, 5, 27), 
    disabled=False
)

end_picker = widgets.DatePicker(
    description="To:",
    value=date(2024, 6, 17),  
    disabled=False
)

output = widgets.Output()
selected_dates = []

def update_range(change=None):
    global selected_dates
    if start_picker.value and end_picker.value:
        start = start_picker.value
        end = end_picker.value
        if start > end:
            with output:
                output.clear_output()
                print("⚠️ The start date must be earlier than or equal to the end date..")
            return
        selected_dates = [(start + timedelta(days=i)).isoformat() 
                          for i in range((end - start).days + 1)]
        with output:
            output.clear_output()
            print("Selected date:", selected_dates)

# Follow pickers
start_picker.observe(update_range, names="value")
end_picker.observe(update_range, names="value")

# ✅ default time range start
update_range()

display(widgets.HBox([start_picker, end_picker]), output)


Output()


## Build AOI geometry and time range variables


In [4]:
from shapely.geometry import box

STAC_URL = "https://stac.eodc.eu/api/v1"
COLLECTION_ID = "GFM"

# tu premeníme selected_bbox na Shapely box
if selected_bbox:
    aoi_geometry = box(
        selected_bbox["minx"],
        selected_bbox["miny"],
        selected_bbox["maxx"],
        selected_bbox["maxy"]
    )
else:
    aoi_geometry = None  # ak ešte nie je nič nakreslené

# časový rozsah z widgetov (zoznam dní v ISO formáte)
if 'selected_dates' in globals() and selected_dates:
    time_range = (selected_dates[0], selected_dates[-1])  # prvý a posledný dátum
else:
    time_range = None  # ak si ešte nič nevybral

print("AOI geometry:", aoi_geometry)
if aoi_geometry:
    print("AOI bounds:", aoi_geometry.bounds)
print("Time range:", time_range)


AOI geometry: POLYGON ((72.594678 31.3511, 72.594678 31.688254, 72.012464 31.688254, 72.012464 31.3511, 72.594678 31.3511))
AOI bounds: (72.012464, 31.3511, 72.594678, 31.688254)
Time range: ('2022-06-12', '2022-07-15')


## STAC search + load
- Loads only the bands you selected.
- Reprojects to EPSG:4326 for downstream steps.
- Prints timings to the notebook output.

In [5]:
# === STAC QUERY ===
client = Client.open(STAC_URL)
search = client.search(
    collections=[COLLECTION_ID],
    intersects=mapping(aoi_geometry),
    datetime=f"{time_range[0]}/{time_range[1]}"
)
items = search.item_collection()
print(f"🔍 Found {len(items)} items.")
if len(items) == 0:
    raise RuntimeError("No STAC items found.")

# CRS & resolution
crs = pyproj.CRS.from_wkt(items[0].properties["proj:wkt2"])
res = items[0].properties.get("gsd", 20.0)
print("Source CRS:", crs)
print("Resolution:", res)

# === LOAD DATA ===
xx = odc.stac.load(
    items,
    crs=crs,
    bbox=aoi_geometry.bounds,
    bands=["ensemble_flood_extent"],
    resolution=res,
    dtype="uint8",
    fail_on_error=False,
)
xx = xx.rio.reproject("EPSG:4326")

🔍 Found 53 items.
Source CRS: PROJCS["Azimuthal_Equidistant",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],AUTHORITY["EPSG","4326"]],PROJECTION["Azimuthal_Equidistant"],PARAMETER["false_easting",4340913.84808],PARAMETER["false_northing",4812712.92347],PARAMETER["longitude_of_center",94.0],PARAMETER["latitude_of_center",47.0],UNIT["metre",1,AUTHORITY["EPSG","9001"]]]
Resolution: 20


## This cell takes a time-series of flood data (one image per observation) and turns it into a map of flooded areas with a count of how many times each area was observed flooded.

- Read a stack of flood images for the selected time period (one image per observation/time step).
- Ignore any “no data” pixels and mark each pixel as flooded or not for each observation.
- Count how many times each pixel was marked flooded across the whole period (this is the number of observations/detections).
- Merge neighbouring flooded pixels into polygons so each continuous flooded patch becomes a single area.
- Build a table where every polygon has a `GFM_observed_flood` value (the count of observations) and a `time_range` string describing the analysed interval.
- Save the result as `observed_flood.geojson` and print a quick summary (how many polygons had 1, 2, 3, … observations).

In [6]:
# source
flood = xx["ensemble_flood_extent"]  # (time, y, x), uint8, nodata=255 (from your debug)

# mask out NoData and make a boolean (flooded / not flooded)
nodata = flood.rio.nodata
flood_masked = flood.where(flood != nodata) if nodata is not None else flood
flood_bool = (flood_masked > 0)

# count number of flooded days (sum over time dimension)
flood = flood_bool.sum(dim="time").astype("uint16")   # (y, x), 0..T

# convert to numpy array for shapes()
arr = flood.data
if hasattr(arr, "compute"):  # dask -> numpy
    arr = arr.compute()
arr = np.ascontiguousarray(arr)

# polygonize only where days > 0 (8-connectivity)
transform = flood.rio.transform()
mask = arr > 0
geoms = shapes(arr, mask=mask, transform=transform, connectivity=8)

# build GeoDataFrame with attribute "days_flooded"
records = [{"geometry": shape(geom), "GFM_observed_flood": int(val)} for geom, val in geoms if val > 0]
gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
gdf["time_range"] = f"{time_range[0]} → {time_range[1]}"

# (optional) small geometry simplification based on resolution, to reduce file size
try:
    res_x, res_y = map(abs, flood.rio.resolution())
    tol = 0.5 * min(res_x, res_y)
except Exception:
    pass

# export to GeoJSON
out_file = "observed_flood.geojson"
gdf.to_file(out_file, driver="GeoJSON", COORDINATE_PRECISION=6)
print(f"✅ Saved {out_file} | features: {len(gdf)}")

# quick check of value distribution (how many polygons with 1 day, 2 days, …)
print(gdf["GFM_observed_flood"].value_counts().sort_index().head(20))


✅ Saved observed_flood.geojson | features: 13738
GFM_observed_flood
1    2699
2    4877
3    2976
4    1581
5     943
6     425
7     143
8      79
9      15
Name: count, dtype: int64


In [7]:
from IPython.display import HTML
HTML("""
<style>
.maplibregl-popup-content { 
  color: #111 !important; 
  background: #fff !important; 
}
.maplibregl-popup-tip { 
  border-top-color: #fff !important; 
}
.maplibregl-popup-close-button { 
  color: #111 !important;
}
</style>
""")


## Loads the observed flood polygons, finds the maximum flood count and creates a dynamic legend and 3D extrusion style on top of Esri World Imagery.

- Loads the flood data from a GeoJSON file.  
- Calculates the highest observed flood value.  
- Chooses colors dynamically based on that maximum value.  
- Draws 3D extrusions of flooded areas on top of Esri World Imagery.  
- Adds a legend that matches the selected time range.


In [8]:
# Dynamic legend + colors based on the maximum value in GFM_observed_flood
import os, json
import leafmap.maplibregl as leafmap

# ==== INPUTS ====
url = "observed_flood.geojson"      # GeoJSON with attribute 'GFM_observed_flood'
ATTR = "GFM_observed_flood"

# ---- AOI (area of interest); safe fallback if not defined earlier
try:
    aoi_geometry
except NameError:
    from shapely.geometry import box
    aoi_geometry = box(10.70, 48.28, 10.98, 48.46)  # Augsburg fallback

minx, miny, maxx, maxy = aoi_geometry.bounds
lon_c = (minx + maxx) / 2
lat_c = (miny + maxy) / 2

# ---- Esri World Imagery basemap (no DEM, no sky)
style_esri = {
    "version": 8,
    "sources": {
        "esri": {
            "type": "raster",
            "tiles": [
                "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
            ],
            "tileSize": 256,
            "attribution": "Source: Esri — World Imagery © Esri, Maxar, Earthstar Geographics, and the GIS User Community",
        }
    },
    "layers": [
        {"id": "esri-ortho", "type": "raster", "source": "esri"}
    ]
}

m = leafmap.Map(
    center=[lat_c, lon_c],  # leafmap maplibregl expects [lat, lon]
    zoom=12,
    pitch=0,
    bearing=0,
    style=style_esri,
)

# ==== DYNAMIC LEGEND AND MAX VALUE DETECTION ====
if not os.path.exists(url):
    print(f"⚠️ File '{url}' not found – only basemap will be displayed.")
else:
    # Load GeoJSON and extract values
    with open(url, "r", encoding="utf-8") as f:
        gj = json.load(f)

    vals = []
    for feat in gj.get("features", []):
        props = feat.get("properties", {})
        v = props.get(ATTR, None)
        try:
            if isinstance(v, str):
                v = float(v.strip())
            v = float(v)
            if v == int(v):
                v = int(v)
            vals.append(v)
        except Exception:
            continue

    # Ensure max is at least 1
    vals = [v for v in vals if v is not None]
    max_val = int(max(vals)) if vals else 1
    if max_val < 1:
        max_val = 1

    # Color palette for 1..5 (if max>5, last color covers 5+)
    palette = ["#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c"]

    # ---- MATCH expression for discrete colors
    color_expr = ["match", ["to-number", ["get", ATTR]]]
    legend_dict = {}

    if max_val <= 5:
        for i in range(1, max_val + 1):
            color_expr.extend([i, palette[i - 1]])
            legend_dict[f"{i} times" if i > 1 else "1 time"] = palette[i - 1]
        # default color (if value outside range)
        color_expr.append(palette[0])
    else:
        for i in range(1, 5):
            color_expr.extend([i, palette[i - 1]])
            legend_dict[f"{i} times" if i > 1 else "1 time"] = palette[i - 1]
        # all >=5 values into last color
        color_expr.extend([5, palette[4]])
        color_expr.append(palette[4])
        legend_dict["5+ times"] = palette[4]

    # ---- Extrusion height
    val_num = ["coalesce", ["to-number", ["get", ATTR]], 0]
    height_expr = ["*", val_num, 6]  # adjust multiplier if needed

    paint_fill = {
        "fill-extrusion-color": color_expr,
        "fill-extrusion-height": height_expr,
        "fill-extrusion-opacity": 0.85,
    }

    # Add flood layer
    m.add_geojson(
        url,
        layer_type="fill-extrusion",
        paint=paint_fill,
        name="Observed flood extent",
        fit_bounds=False,
    )

    # ==== LEGEND TITLE (dynamic based on widgets, if available) ====
    try:
        start_date = start_picker.value
        end_date = end_picker.value
        if start_date and end_date:
            title = f"Observed floods by satellite<br>{start_date.isoformat()} → {end_date.isoformat()}"
        else:
            title = "Observed floods by satellite<br>(no range selected)"
    except NameError:
        title = "Observed floods by satellite<br>(no range selected)"

    m.add_legend(title=title, legend_dict=legend_dict, position="bottom-right")

# Camera (slight tilt for effect)
if hasattr(m, "fly_to"):
    m.fly_to(lon=lon_c, lat=lat_c, zoom=12, pitch=45, bearing=15, duration=4500)
else:
    m.fit_bounds([[minx, miny], [maxx, maxy]])
    if hasattr(m, "set_pitch"):
        m.set_pitch(45)

m


Container(children=[Row(children=[Col(children=[Col(children=[Map(calls=[['addControl', ('NavigationControl', …